In [ ]:
# @title Required modules, during initialization start execution then restart session and do not run this block again
%pip install accelerate -U
%pip install datasets -q
%pip install transformers -q
%pip install pandas -q
%pip install nltk -q
%pip install torch -q

In [ ]:
# @title Imports required modules and mounts google drive
import nltk
from transformers import AutoTokenizer, AutoModelForTokenClassification, Trainer, TrainingArguments, DistilBertTokenizerFast
from google.colab import drive
import pandas as pd
from nltk.tokenize import TreebankWordTokenizer as twt
import datasets
from datasets import DatasetDict, Dataset, concatenate_datasets
import numpy as np
import torch
from torch.utils.data import Subset
import time
import os
import random
from google.colab import data_table
from sklearn.metrics import confusion_matrix, f1_score, accuracy_score, recall_score,precision_score
import math

nltk.download('punkt')
#drive.mount('/content/drive')

In [ ]:
# @title Tag Relation Mapping and global configurations
tag_relation_mapping = {
    'Ekosistem': {'type': 'top', 'hypernyms': [], 'hyponyms': ['Karasal Ekosistem', 'Yerleşim Yerleri', 'Sucul Ekosistem'], 'siblings': []},
    'Karasal Ekosistem': {'type': 'middle', 'hypernyms': ['Ekosistem'], 'hyponyms': ['Yerleşim Yerleri'], 'siblings': ['Sucul Ekosistem']},
    'Yerleşim Yerleri': {'type': 'bottom', 'hypernyms': ['Ekosistem', 'Karasal Ekosistem'], 'hyponyms': [], 'siblings': []},
    'Sucul Ekosistem': {'type': 'bottom', 'hypernyms': ['Ekosistem'], 'hyponyms': [], 'siblings': ['Karasal Ekosistem']},
    'Kirletici': {'type': 'top', 'hypernyms': [], 'hyponyms': ['Sıvı Kirletici', 'Katı Kirletici', 'Gaz Kirletici',  'Enerji'], 'siblings': []},
    'Sıvı Kirletici': {'type': 'bottom', 'hypernyms': ['Kirletici'], 'hyponyms': [], 'siblings': ['Katı Kirletici', 'Gaz Kirletici', 'Enerji']},
    'Katı Kirletici': {'type': 'bottom', 'hypernyms': ['Kirletici'], 'hyponyms': [], 'siblings': ['Sıvı Kirletici', 'Gaz Kirletici', 'Enerji']},
    'Gaz Kirletici': {'type': 'bottom', 'hypernyms': ['Kirletici'], 'hyponyms': [], 'siblings': ['Sıvı Kirletici', 'Katı Kirletici', 'Enerji']},
    'Enerji': {'type': 'bottom', 'hypernyms': ['Kirletici'], 'hyponyms': [], 'siblings': ['Sıvı Kirletici', 'Katı Kirletici', 'Gaz Kirletici']},
    'Afet': {'type': 'top', 'hypernyms': [], 'hyponyms': ['Doğal Afet', 'İnsan Kaynaklı Afet'], 'siblings': []},
    'Doğal Afet': {'type': 'bottom', 'hypernyms': ['Afet'], 'hyponyms': [], 'siblings': ['İnsan Kaynaklı Afet']},
    'İnsan Kaynaklı Afet': {'type': 'bottom', 'hypernyms': ['Afet'], 'hyponyms': [], 'siblings': ['Doğal Afet']},
    'Biota': {'type': 'top', 'hypernyms': [], 'hyponyms': ['İnsan Dışı Biota', 'Sucul Biota', 'Karasal Biota', 'İnsan'], 'siblings': []},
    'İnsan Dışı Biota': {'type': 'middle', 'hypernyms': ['Biota'], 'hyponyms': ['Sucul Biota', 'Karasal Biota'], 'siblings': ['İnsan']},
    'Sucul Biota': {'type': 'bottom', 'hypernyms': ['İnsan Dışı Biota', 'Biota'], 'hyponyms': [], 'siblings': ['Karasal Biota']},
    'Karasal Biota': {'type': 'bottom', 'hypernyms': ['İnsan Dışı Biota', 'Biota'], 'hyponyms': [], 'siblings': ['Sucul Biota']},
    'İnsan': {'type': 'bottom', 'hypernyms': ['Biota'], 'hyponyms': [], 'siblings': ['İnsan Dışı Biota']},
    'Çevresel Etki': {'type': 'top', 'hypernyms': [], 'hyponyms': ['Ekolojik Etki', 'Refah Etkisi', 'Ekonomik Etki', 'Sağlık Etkisi', 'Sosyal Etki'], 'siblings': []},
    'Ekolojik Etki': {'type': 'bottom', 'hypernyms': ['Çevresel Etki'], 'hyponyms': [], 'siblings': ['Refah Etkisi']},
    'Refah Etkisi': {'type': 'middle', 'hypernyms': ['Çevresel Etki'], 'hyponyms': ['Ekonomik Etki', 'Sağlık Etkisi', 'Sosyal Etki'], 'siblings': ['Ekolojik Etki']},
    'Ekonomik Etki': {'type': 'bottom', 'hypernyms': ['Refah Etkisi', 'Çevresel Etki'], 'hyponyms': [], 'siblings': ['Sağlık Etkisi',  'Sosyal Etki']},
    'Sağlık Etkisi': {'type': 'bottom', 'hypernyms': ['Refah Etkisi', 'Çevresel Etki'], 'hyponyms': [], 'siblings': ['Ekonomik Etki',  'Sosyal Etki']},
    'Sosyal Etki': {'type': 'bottom', 'hypernyms': ['Refah Etkisi', 'Çevresel Etki'], 'hyponyms': [], 'siblings': ['Ekonomik Etki',  'Sağlık Etkisi']},
    'Çevre Yönetimi': {'type': 'top', 'hypernyms': [], 'hyponyms': ['Düzenleme', 'Azaltma', 'Arıtım'], 'siblings': []},
    'Düzenleme': {'type': 'bottom', 'hypernyms': ['Çevre Yönetimi'], 'hyponyms': [], 'siblings': ['Azaltma', 'Arıtım']},
    'Azaltma': {'type': 'bottom', 'hypernyms': ['Çevre Yönetimi'], 'hyponyms': [], 'siblings': ['Düzenleme', 'Arıtım']},
    'Arıtım': {'type': 'bottom', 'hypernyms': ['Çevre Yönetimi'], 'hyponyms': [], 'siblings': ['Düzenleme', 'Azaltma']},
    'Kirleten': {'type': 'top', 'hypernyms': [], 'hyponyms': ['İnsan Kaynaklı Kirleten',  'Doğal Kirleten'], 'siblings': []},
    'İnsan Kaynaklı Kirleten': {'type': 'bottom', 'hypernyms': ['Kirleten'], 'hyponyms': [], 'siblings': ['Doğal Kirleten']},
    'Doğal Kirleten': {'type': 'bottom', 'hypernyms': ['Kirleten'], 'hyponyms': [], 'siblings': ['İnsan Kaynaklı Kirleten']}
}
version = 101
global_seed = 16 + version
random.seed(global_seed)

In [ ]:
# @title Creates configurations
# labels with no hypernyms
top_level_labels = []
# labels that have hypernyms and hyponyms
mid_level_labels = []
# labels with no hyponyms
bottom_level_labels = []
# labels with siblings
sibling_labels = []
# labels without siblings
no_sibling_labels = []

for tag in tag_relation_mapping:
  if len(tag_relation_mapping[tag]['hypernyms']) == 0:
    top_level_labels.append(tag)
  elif len(tag_relation_mapping[tag]['hyponyms']) == 0:
    bottom_level_labels.append(tag)
  else:
    mid_level_labels.append(tag)

  if len(tag_relation_mapping[tag]['siblings']) > 0:
    sibling_labels.append(tag)
  else:
    no_sibling_labels.append(tag)

def get_relation_configurations(tag):
                  # Hyponym, hypernym, sibling
  if tag in bottom_level_labels:
    if tag in sibling_labels:
      return [[False, False, True]]
    else:
      return None

  conf = [[False, False, False]]

  if tag in top_level_labels:
    conf.append([True, False, False])

  if tag in mid_level_labels:
    conf.append([True, False, False])
    conf.append([False, True, False])
    conf.append([True, True, False])

  if tag in sibling_labels:
    conf.append([False, False, True])

  return conf




In [ ]:
# @title RawDataToHuggingfaceDatasetDictConverter creates huggingface dataset dict from given jsonl file path
file_path = "/content/drive/MyDrive/thesis_dataset/news_and_gpt_dataset_preprocessed.jsonl" # @param {type:"string"}
#file_path = "/content/drive/MyDrive/thesis_dataset/news_only_dataset.jsonl" # @param {type:"string"}
class RawDataToHuggingfaceDatasetDictConverter():
  def __init__(self, file_path):
    self.file_path = file_path

  def convert(self):
    json_line_file = self._read_json_line_file()
    ds = self._create_huggingface_dataset(json_line_file)
    self.dataset = self._augment_dataset(ds)
    df = pd.DataFrame(raw_data_converter.dataset.to_pandas().groupby(['tag', 'tokens_str'], group_keys=False).apply(self._merge_same_tags))
    df = df.reset_index(level='tag', drop=True)
    df = df.drop(columns= ['tokens_str'])
    df = df.reset_index(drop = True)
    self.dataset = Dataset.from_pandas(df)

  def count_df(self):
    df = self.dataset.to_pandas()['tag'].value_counts().rename_axis('tag_name').reset_index(name='counts')
    df['percent'] = (df['counts'] / df['counts'].sum()) * 100
    return df

  def _merge_same_tags(self, rows):
    if len(rows) == 1:
      return rows.iloc[0]
    else:
      grouped_tags = rows.iloc[0]['tags']
      for i in range(1, len(rows)):
        row = rows.iloc[i]
        for j in range(len(row.loc['tags'])):
          if row.loc['tags'][i] != 'O':
            grouped_tags[i] = row.loc['tags'][i]
      rows.iloc[0]['tags'] = grouped_tags

      return rows.iloc[0]


  def _augment_dataset(self, ds):
    pre_augmented_ds = ds.map(self._expand)['row']
    augmented_list = []
    for expanded in pre_augmented_ds:
      for row in expanded:
        augmented_list.append(row)
    return Dataset.from_list(augmented_list)

  def _expand(self, example):
    tag = example['tag']
    tokens = example['tokens'],
    tags = example['tags']
    hypernyms = tag_relation_mapping[tag]['hypernyms']
    expanded = [{
      'tokens_str': f'{tokens}',
      'tag': tag,
      'tokens': tokens,
      'tags': tags
    }]
    if hypernyms is not None and len(hypernyms) > 0:
      for hypernym in hypernyms:
        expanded.append({
            'tokens_str': f'{tokens}',
            'tag': hypernym,
            'tokens': tokens,
            'tags': [hypernym if x == tag else x for x in tags],
        })
    return {'row': expanded}


  def _read_json_line_file(self):
    return pd.read_json(path_or_buf=self.file_path, lines=True)

  def _create_huggingface_dataset(self, jsonArr):
    self.twt = twt()
    self.mappedArr = jsonArr.apply(self._json_to_token_tag_pair, axis = 1)
    all = self._flat_mapped_arr(self.mappedArr)
    df = pd.DataFrame.from_records(data=all)
    ds = datasets.Dataset.from_pandas(df)
    return ds


  def _flat_mapped_arr(self, mappedArr):
    all = []
    for t in mappedArr:
      for e in t:
        all.append(e)
    return all


  def _get_unique_entities(self, entities):
    entity_obj = {}
    for entity in entities:
      if entity['label'] not in entity_obj:
        entity_obj[entity['label']] = []

      entity_obj[entity['label']].append({'start_offset': entity['start_offset'], 'end_offset': entity['end_offset']})
    return entity_obj

  def _make_pair(self, text, spans, entities):
    unique_entities = self._get_unique_entities(entities)

    all_tags = []
    for entity in unique_entities:
      attention=[]
      tokens=[]
      entity_offsets = unique_entities[entity]
      for start,end in spans:
        word = text[start:end]
        attention.append(word)
        tag_found = False
        for offset in entity_offsets:
          if start >= offset['start_offset'] and end <= offset['end_offset']:
            tokens.append(entity)
            tag_found = True
            break
        if not tag_found:
          tokens.append('O')
      all_tags.append({'tags': tokens, 'tokens': attention, 'tag': entity})
    return all_tags

  def _json_to_token_tag_pair(self, row):
    spans = list(self.twt.span_tokenize(row['text']))
    return self._make_pair(row['text'], spans, row['entities'])


In [ ]:
# @title LoaderParamsCreator creates loader params for different classes, shot options and other options
#model_checkpoint = "/content/drive/MyDrive/Results/TurkishNER/ultimate_19-03-2024/" # @param {type:"string"}
model_checkpoint = "dbmdz/distilbert-base-turkish-cased" # @param {type:"string"}
tokenizer_checkpoint = 'dbmdz/distilbert-base-turkish-cased' # @param {type:"string"}
use_gpu = True # @param {type:"boolean"}
class LoaderParamsCreator():
  def __init__(self, tag_relation_mapping, converter, model_checkpoint, tokenizer_checkpoint, use_gpu):
    self.tag_relation_mapping = tag_relation_mapping
    self.converter = converter
    self.model_checkpoint = model_checkpoint
    self.tokenizer_checkpoint = tokenizer_checkpoint
    self.use_gpu = use_gpu
    self._create_loader_params_list()

  def _find_hyponyms(self, tag):
    return self.tag_relation_mapping[tag]['hyponyms']

  def _find_hypernyms(self, tag):
    return self.tag_relation_mapping[tag]['hypernyms']

  def _find_siblings(self, tag):
    return self.tag_relation_mapping[tag]['siblings']

  def _find_label_count(self, count_df, label):
    counts = count_df[count_df['tag_name']==label]['counts']
    return counts.values[0] if len(counts) > 0 else 0

  def _find_labels_count(self, count_df, labels):
    return sum([self._find_label_count(count_df, label) for label in labels])

  def _create_loader_params_list(self):
    count_df = self.converter.count_df()
    ner_labels = count_df['tag_name']
    shot_options = [0, 1, 10, 100]

    loader_params_dict = {}
    loader_params_list = []
    for ner_label in ner_labels:
      # Hyponym, hypernym, sibling
      remove_relation_configs = get_relation_configurations(ner_label)
      if remove_relation_configs is None:
        continue
      print(ner_label, remove_relation_configs)
      for shot_option in shot_options:
        label_count = self._find_label_count(count_df, ner_label)
        if label_count <= 3 or label_count <= shot_option * 3:
          continue

        for remove_relation_config in remove_relation_configs:
          hidden_labels = []
          hyponyms = self._find_hyponyms(ner_label)
          hyponyms_count = self._find_labels_count(count_df, hyponyms) if len(hyponyms) > 0 else 0
          hypernyms = self._find_hypernyms(ner_label)
          hypernyms_count = self._find_labels_count(count_df, hypernyms) if len(hypernyms) > 0 else 0
          siblings = self._find_siblings(ner_label)
          siblings_count = self._find_labels_count(count_df, siblings) if len(siblings) > 0 else 0

          current_config = [
              hyponyms_count == 0 or remove_relation_config[0],
              hypernyms_count == 0 or remove_relation_config[1],
              siblings_count == 0 or remove_relation_config[2]
          ]

          param_key = str(current_config) + str(shot_option) + str(ner_label)
          if param_key in loader_params_dict:
            continue

          if remove_relation_config[0]:
            hidden_labels.extend(self._find_hyponyms(ner_label))
          if remove_relation_config[1]:
            hidden_labels.extend(self._find_hypernyms(ner_label))
          if remove_relation_config[2]:
            hidden_labels.extend(self._find_siblings(ner_label))



          loader_param =  {
            'use_gpu': self.use_gpu,
            'model_checkpoint': self.model_checkpoint,
            'tokenizer_checkpoint': tokenizer_checkpoint,
            'class_unseen': ner_label,
            'shot_number': shot_option,
            'hidden_labels': hidden_labels,
            'hyponyms': hyponyms,
            'hypernyms': hypernyms,
            'siblings': siblings,
            'removed_hyponyms': current_config[0],
            'removed_hypernyms': current_config[1],
            'removed_siblings': current_config[2]
          }

          loader_params_dict[param_key] = loader_param

          loader_params_list.append(loader_param)

    self.loader_params_list = loader_params_list
    self.loader_params_dict = loader_params_dict

  def get_loader_params_df(self):
    return pd.DataFrame(self.loader_params_list)

  def get_loaders_params_dict_as_df(self):
    df = pd.DataFrame(self.loader_params_dict.values())
    return df

In [ ]:
# @title HuggingfaceToTorchConverter creates torch dataset from huggingface dataset
class HuggingfaceToTorchConverter(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

In [ ]:
# @title DataLoader gets huggingface dict from converter and splits data according to given shot number, also hides given labels from the dataset
class DataLoader():
    def __init__(self, converter, use_gpu, model_checkpoint, tokenizer_checkpoint, shot_number, hidden_labels, class_unseen = None, **kwargs) -> None:
        self.converter = converter
        self.use_gpu = use_gpu
        self.model_checkpoint = model_checkpoint
        self.tokenizer_checkpoint = tokenizer_checkpoint
        self.class_unseen = class_unseen
        self.hidden_labels = hidden_labels
        self.shot_number = shot_number

    def create_data(self):
        self.tokenizer = AutoTokenizer.from_pretrained(self.tokenizer_checkpoint)
        self.dataset_dict = self._train_test_validation_split(self.converter.dataset)
        self.df_train = self._prepare_data(self.dataset_dict['train'])
        self.df_test = self._prepare_data(self.dataset_dict['test'])
        self.df_valid = self._prepare_data(self.dataset_dict['validation'])

        self.tokenized_encodings_train = self._BERTTokenization_ClassText(self.df_train)
        self.tokenized_encodings_test = self._BERTTokenization_ClassText(self.df_test)
        self.tokenized_encodings_valid = self._BERTTokenization_ClassText(self.df_valid)

        self.labels_train = self._align_labels(self.df_train, self.tokenized_encodings_train, True)
        self.labels_test = self._align_labels(self.df_test, self.tokenized_encodings_test, True)
        self.labels_valid = self._align_labels(self.df_valid, self.tokenized_encodings_valid, True)

        self.dataset_train = HuggingfaceToTorchConverter(self.tokenized_encodings_train, self.labels_train)
        self.dataset_test = HuggingfaceToTorchConverter(self.tokenized_encodings_test, self.labels_test)
        self.dataset_valid = HuggingfaceToTorchConverter(self.tokenized_encodings_valid, self.labels_valid)

    def _train_test_validation_split(self, dataset):
        #filtered_dataset = dataset.filter(lambda x : x['tag'] not in self.hidden_labels) if len(self.hidden_labels) else dataset
        filtered_dataset = dataset

        test_dataset_raw = filtered_dataset.filter(lambda x : x['tag'] == self.class_unseen)
        train_dataset_raw = filtered_dataset.filter(lambda x : x['tag'] != self.class_unseen)

        train_and_valid_dataset = train_dataset_raw.train_test_split(test_size=0.15, seed=global_seed)
        train_dataset = train_and_valid_dataset['train']
        valid_dataset = train_and_valid_dataset['test']

        test_dataset_raw = test_dataset_raw.shuffle(seed=global_seed)
        print("Length class_unseen:", len(test_dataset_raw))

        if (len(test_dataset_raw) / 3) < 1:
          max_shot_number = 0
        elif len(test_dataset_raw) / 3 < 10:
          max_shot_number = 1
        else:
          max_shot_number = 10

        if self.shot_number > max_shot_number:
            return

        test_valid_split_index = math.floor((len(test_dataset_raw) + max_shot_number) / 2)

        unseen_train_fixed = test_dataset_raw.select(range(max_shot_number)) if max_shot_number > 0 else None
        #unseen_validation_fixed = test_dataset_raw.select(range(max_shot_number, test_valid_split_index))
        unseen_test_fixed = test_dataset_raw.select(range(max_shot_number, len(test_dataset_raw)))

        if self.shot_number > 0:
            train_dataset = concatenate_datasets([train_dataset, unseen_train_fixed.select(range(self.shot_number))])

        if len(self.hidden_labels) > 0:
          train_dataset = train_dataset.filter(lambda x : x['tag'] not in self.hidden_labels)
          valid_dataset = valid_dataset.filter(lambda x : x['tag'] not in self.hidden_labels)

        print("unseen_train_fixed:", len(unseen_train_fixed))
        #print("unseen_validation_fixed:", len(unseen_validation_fixed))
        print("unseen_test_fixed:", len(unseen_test_fixed))

        return DatasetDict({
          'train': train_dataset,
          'validation': valid_dataset,
          'test': unseen_test_fixed
        })


    def _split_labels(self, example, id):
        m_arr = []
        tags = np.array(example['tags'])
        text = example['tokens']
        cls = example['tag']
        cor = "Kerem_ENV"
        labels = []
        for tag in tags:
          if tag == cls:
            labels.append(1)
          else:
            labels.append(0)
        m_arr.append({
          'id': id,
          'cor': cor,
          'class': [cls],
          'text': text[0],
          'labels': labels,
          'klasa': cls,
          'tags': tags
        })
        return m_arr

    def _prepare_data(self, ds):
        data_arr = []
        for _, example in enumerate(ds):
            data_arr.append(self._split_labels(example, len(data_arr) + 1))
        np_arr = np.array(data_arr)
        np_arr = np_arr.flatten()
        return pd.DataFrame(list(np_arr))


    def _BERTTokenization_ClassText(self, df):
        tokenized_encodings = self.tokenizer(df["class"].to_list(),
                                 df["text"].to_list(),
                                 truncation=True,
                                 is_split_into_words=True,
                                 add_special_tokens=True,
                                 padding='longest',
                                 max_length=512)
        return tokenized_encodings


    def _align_labels(self, df, tokenized_encodings, label_all_tokens = True):
        labels = list()
        for i, label in enumerate(df['labels']):
            word_ids = tokenized_encodings.word_ids(batch_index=i)  # Map tokens to their respective word.
            previous_word_idx = None
            label_ids = []
            for word_idx in word_ids:  # Set the special tokens to -100.
                if word_idx is None:
                    label_ids.append(-100)
                elif word_idx != previous_word_idx:
                    if word_idx < len(label):
                     label_ids.append(label[word_idx])
                    else:
                        label_ids.append(-100)
                else:
                    label_ids.append(label[word_idx] if label_all_tokens else -100)
                previous_word_idx = word_idx
            stop = label_ids.index(-100, 2) # the second occurrence of None (-100) is on this index
            label_ids = label_ids[:1] + [1 for x in label_ids[1:stop]] + label_ids[stop:]
            labels.append(label_ids)
        return labels


    def hides_Dataset_class_returns_TrainValidTest_Subsets(self):

        if self.shot_number == 0:
            indexes_train = list(self.df_train.index[self.df_train['klasa']!=self.class_unseen])
            indexes_test = list(self.df_test.index[self.df_test['klasa']==self.class_unseen])

            self.ds_train = Subset(self.dataset_train, indexes_train)
            self.ds_test = Subset(self.dataset_test, indexes_test)
        else:
            lst_indexes_FewShot_train = list(self.df_train.index[(self.df_train['klasa']==self.class_unseen)
                                                             & (self.df_train['labels'].apply(lambda lst : sum(lst))>0)])
            indexes_UnseenClass_train = random.choices(lst_indexes_FewShot_train, k=self.shot_number)
            self.ds_train = Subset(self.dataset_train, indexes_UnseenClass_train)

In [ ]:
# @title TrainTestDriver
class TrainTestDriver():
  def __init__(self, raw_data_converter, loader_params):
    self.loader_params = loader_params
    self.raw_data_converter = raw_data_converter
    self.data_loader = DataLoader(raw_data_converter, **self.loader_params)
    self.data_loader.create_data()

  def training_info(self):
    loader_params_series = pd.Series(self.loader_params)
    dataset_info = pd.Series({
        'train_length': len(self.data_loader.dataset_train),
        'validation_length': len(self.data_loader.dataset_valid),
        'test_length': len(self.data_loader.dataset_test),
        'total_length': len(self.data_loader.dataset_train)+len(self.data_loader.dataset_valid)+len(self.data_loader.dataset_test)
      })
    return pd.concat([loader_params_series, dataset_info, self.training_results, self.test_results])


  def create_trainer(self):
    print('loader params', self.loader_params)
    self.training_args = TrainingArguments(
      output_dir='./Results'+self.loader_params['class_unseen']+'ZeroShot',   # output folder (folder to store the results)
      num_train_epochs=6,                               # number of training epochs
      per_device_train_batch_size=16,                   # batch size per device during training
      per_device_eval_batch_size=16,                    # batch size for evaluation
      weight_decay=0.01,                                # strength of weight decay
      logging_dir='./Logs'+self.loader_params['class_unseen']+'ZeroShot',     # folder to store the logs
      logging_steps=10,
      logging_strategy='steps',
      save_strategy='steps',
      save_steps=500,
      evaluation_strategy='steps',
      eval_steps=500,
      save_total_limit=1,
      load_best_model_at_end=True
    )
    self.model = AutoModelForTokenClassification.from_pretrained(self.loader_params['model_checkpoint'], num_labels=2)
    if self.loader_params['use_gpu']:
      self.model.to(torch.device('cuda'))
    self.trainer = Trainer(
        model=self.model,                # pre-trained model for fine-tuning
        args=self.training_args,          # training arguments defined above
        train_dataset=self.data_loader.dataset_train,   # dataset class object for training
        eval_dataset=self.data_loader.dataset_valid   # dataset class object for validation
    )

  def train(self):
    start_time = time.time()
    training_results = self.trainer.train()
    self.training_results = pd.Series(training_results.metrics)
    self.training_time = time.time()-start_time


  def test(self):
    test_results = self.__test(self.data_loader.dataset_test, self.model, self.data_loader.df_test)
    self.test_results = pd.Series(test_results)


  def __test(self, testset, model, df_test):
    args = TrainingArguments(output_dir='./evaldir', per_device_eval_batch_size=16)

    evaluator = Trainer(
        args=args,
        model=model
    )

    pred=evaluator.predict(testset)
    test_indexs_new=df_test.index.to_list()
    wids=np.array([testset.encodings.encodings[ii].word_ids for ii in test_indexs_new])

    wids[wids==None]=-1
    wids=wids.astype(int)
    type_ids=np.array([testset.encodings.encodings[ii].type_ids for ii in test_indexs_new],dtype=bool)
    pre=pred[0].argmax(axis=-1)
    pre_list=[]
    test_list=[]
    for ii in range(wids.shape[0]):
        test_list.append(wids[ii][type_ids[ii]])
        pre_list.append(pre[ii][type_ids[ii]])

    labels=[]
    for ii in range(len(pre_list)):
        labels.append(np.array(range(test_list[ii].max()+1)))
        for jj in labels[ii]:
            bb=np.where(test_list[ii]==jj)[0]
            labela=np.array(pre_list[ii])[bb].mean()
            if labela>0.01:
                labels[ii][jj]=1
            else:
                labels[ii][jj]=0

    labels_original=list(df_test.iloc[:]['labels'])
    f1av=0.0
    lf1=0

    for ii in range(len(labels)):
        if len(labels[ii])==len(labels_original[ii]):
            f1av=f1av+f1_score( labels_original[ii], labels[ii],average=None)
            lf1=lf1+1
        else:
            print(ii, len(labels[ii]), len(labels_original[ii]))

    lab=np.array([])       # predicted labels
    labor=np.array([])     # true labels labele


    for ii in range(len(labels)):
        if len(labels[ii])==len(labels_original[ii]):
            lab=np.concatenate((lab,labels[ii]))
            labor=np.concatenate((labor,labels_original[ii]))

    accuracy = accuracy_score(labor, lab, normalize=True)

    precision_avg_none = precision_score(labor, lab, average=None)
    precision_avg_binary = precision_score(labor, lab, average='binary')
    precision_avg_micro = precision_score(labor, lab, average='micro')
    precision_avg_macro = precision_score(labor, lab, average='macro')
    precision_avg_weighted = precision_score(labor, lab, average='weighted')

    recall_avg_none = recall_score(labor, lab, average=None)
    recall_avg_binary = recall_score(labor, lab, average='binary')
    recall_avg_micro = recall_score(labor, lab, average='micro')
    recall_avg_macro = recall_score(labor, lab, average='macro')
    recall_avg_weighted = recall_score(labor, lab, average='weighted')

    f1_avg_none = f1_score(labor, lab, average=None)
    f1_avg_binary = f1_score(labor, lab, average='binary')
    f1_avg_micro = f1_score(labor, lab, average='micro')
    f1_avg_macro = f1_score(labor, lab, average='macro')
    f1_avg_weighted = f1_score(labor, lab, average='weighted')

    matrix = confusion_matrix(labor, lab)

    return {
        'accuracy': accuracy,
        'precision_avg_none': precision_avg_none,
        'precision_avg_binary': precision_avg_binary,
        'precision_avg_micro': precision_avg_micro,
        'precision_avg_macro': precision_avg_macro,
        'precision_avg_weighted': precision_avg_weighted,
        'recall_avg_none': recall_avg_none,
        'recall_avg_binary': recall_avg_binary,
        'recall_avg_micro': recall_avg_micro,
        'recall_avg_macro': recall_avg_macro,
        'recall_avg_weighted': recall_avg_weighted,
        'f1_avg_none': f1_avg_none,
        'f1_avg_binary': f1_avg_binary,
        'f1_avg_micro': f1_avg_micro,
        'f1_avg_macro': f1_avg_macro,
        'f1_avg_weighted': f1_avg_weighted,
        'f1_var': f1av/lf1,
        'matrix': matrix
    }

In [ ]:
raw_data_converter = RawDataToHuggingfaceDatasetDictConverter(file_path)
raw_data_converter.convert()
print(raw_data_converter.dataset)
data_table.DataTable(raw_data_converter.count_df(), include_index=True, num_rows_per_page=10)

In [ ]:
loader_params_creator = LoaderParamsCreator(tag_relation_mapping, raw_data_converter, model_checkpoint, tokenizer_checkpoint, use_gpu)
loader_params_count = len(loader_params_creator.loader_params_list)
print(f'Loader Params Count: {loader_params_count}')
print(f'Loader Params Count: {len(loader_params_creator.loader_params_dict)}')

In [ ]:
ind = 83
print(loader_params_creator.loader_params_list[ind])
trainer = TrainTestDriver(raw_data_converter, loader_params_creator.loader_params_list[ind])
print(len(trainer.data_loader.dataset_train))
print(len(trainer.data_loader.dataset_valid))
print(len(trainer.data_loader.dataset_test))
trainer.data_loader.tokenizer.decode(trainer.data_loader.dataset_test[0]['input_ids'])

In [ ]:
sentence_dict = {}
for ind, loader_params in enumerate(loader_params_creator.loader_params_list):
  print(f'Training Index: {ind} / {len(loader_params_creator.loader_params_list)}')
  print(loader_params_creator.loader_params_list[ind])
  trainer = TrainTestDriver(raw_data_converter, loader_params_creator.loader_params_list[ind])
  print(len(trainer.data_loader.dataset_train))
  print(len(trainer.data_loader.dataset_valid))
  print(len(trainer.data_loader.dataset_test))
  for x in trainer.data_loader.dataset_test:
    print(trainer.data_loader.tokenizer.decode(x['input_ids']))
    print(x['labels'])
    sentence_dict[trainer.data_loader.tokenizer.decode(x['input_ids'])] = x['labels']


In [ ]:
print(len(sentence_dict))
sentence_dict

In [ ]:
trainer.data_loader.dataset_test[0]

In [ ]:
#final_list = loader_params_creator.loader_params_list
#for i, loader_params in enumerate(final_list):
#  print(f'Training Index: {i} / {len(final_list)}')
#  print(loader_params)
#  trainer = TrainTestDriver(raw_data_converter,loader_params)
#  print(len(trainer.data_loader.dataset_train))
#  print(len(trainer.data_loader.dataset_valid))
#  print(len(trainer.data_loader.dataset_test))
#  trainer.data_loader.tokenizer.decode(trainer.data_loader.dataset_test[0]['input_ids'])

In [ ]:
loader_params_creator.get_loaders_params_dict_as_df()

In [ ]:
loader_params_list = list(loader_params_creator.loader_params_dict.values())

In [ ]:
ner_results = []

In [ ]:
#final_list = list(filter(lambda x : x['class_unseen'] == 'Karasal Ekosistem', loader_params_list))
#final_list = loader_params_list[:math.floor(len(loader_params_list) / 4)]
final_list = loader_params_list
for i, loader_params in enumerate(final_list):
  print(f'Training Index: {i} / {len(final_list)}')
  train_test_driver = TrainTestDriver(raw_data_converter, loader_params)
  train_test_driver.create_trainer()
  train_test_driver.train()
  train_test_driver.test()
  ner_results.append(train_test_driver.training_info())

In [ ]:
results_df = pd.DataFrame(ner_results).drop(['use_gpu', 'train_samples_per_second', 'total_flos', 'epoch'], axis=1)

In [ ]:
results_df

In [ ]:
from datetime import datetime

datetime.now().strftime("%d-%m-%Y-%H-%M")

In [ ]:
from datetime import datetime

now = datetime.now().strftime("%d-%m-%Y-%H-%M")
model_name = model_checkpoint.split('/')[-1]

csv_name = f'{model_name}_{len(loader_params_creator.loader_params_list)}_{now}_seed_{global_seed}_version_{version}_topandmiddlewithsibling_bottomonlysibling.csv'
results_df.to_csv(f'/content/drive/MyDrive/ner_results/{csv_name}', index=False)